# Study 870 — Industry-Leader Lead-Lag — the teardown

The weekly spread Newey-West *t*, the pooled Welch leg test, the 1,000-permutation placebo, the two-era robustness cut, the dollar-volume leader re-designation, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_weeks': 859, 'spread_bps': -3.64, 't_nw': -0.77, 't_1s': -0.69, 'up_bps': 28.79, 'dn_bps': 35.99, 'welch_t': -1.99, 'gross_sharpe': -0.17, 'placebo_obs': -3.64, 'placebo_mean': 3.465, 'placebo_sd': 4.935, 'placebo_p': 0.928, 'placebo_draws': 1000, 'era_early_bps': -13.96, 'era_early_t': -2.16, 'era_early_n': 415, 'era_late_bps': 5.96, 'era_late_t': 0.89, 'era_late_n': 443, 'dyn_bps': -3.87, 'dyn_t': -0.81, 'timer_1_gross': -3.64, 'timer_1_cost': 2.96, 'timer_1_net': -6.6, 'timer_1_t': -1.24, 'timer_5_gross': -3.64, 'timer_5_cost': 10.96, 'timer_5_net': -14.6, 'timer_5_t': -2.75, 'null_mean_t': -0.04, 'null_sd_t': 0.79, 'null_fire': 0, 'planted_t': 20.79, 'planted_welch': 17.91}

## The headline — long up-leader / short down-leader followers

Weekly equal-per-sector spread: `mean_s sign(leader_w) · mean_followers(ret_{w+1})`.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/week  NW(6) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}  (n = {R['n_weeks']} weeks)")
print(f"legs          : after up-leader {R['up_bps']:+.2f} vs after down-leader {R['dn_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -3.64 bps/week  NW(6) t = -0.77  one-sample t = -0.69  (n = 859 weeks)
legs          : after up-leader +28.79 vs after down-leader +35.99 bps (Welch t = -1.99)
gross Sharpe  : -0.17 (before cost)


## Placebo — shuffle the lead→lag week alignment (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.3f}")
print('the null sits ABOVE the observed value — nothing special in the true alignment')

observed -3.64 bps vs placebo mean +3.465 (sd 4.935) -> right-tail p = 0.928
the null sits ABOVE the observed value — nothing special in the true alignment


## Robustness — two eras (split 2018-01-01) and a dollar-volume leader re-designation

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print(f"$-vol leaders   : {R['dyn_bps']:+.2f} bps  NW t = {R['dyn_t']:+.2f} (same non-result)")
print('sign FLIPS across eras and clears |t|>=2 only in the WRONG direction')

2010-2017 (n=415): -13.96 bps  NW t = -2.16
2018-2026 (n=443): +5.96 bps  NW t = +0.89
$-vol leaders   : -3.87 bps  NW t = -0.81 (same non-result)
sign FLIPS across eras and clears |t|>=2 only in the WRONG direction


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per week on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/week (cost {c:.2f}/wk, t={t:+.2f})")

 1 bp one-way: gross -3.64 -> net -6.60 bps/week (cost 2.96/wk, t=-1.24)
5 bps one-way: gross -3.64 -> net -14.60 bps/week (cost 10.96/wk, t=-2.75)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted diffusion.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from leader_lag import data, strategy as st
secs, lds = data.synthetic_sectors(), data.synthetic_leaders()
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=870+s, n_weeks=200), secs, lds)['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.6, seed=870, n_weeks=320), secs, lds)
print(f"planted (edge=0.6): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of

C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of

C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of

C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of

null (edge=0), 8 seeds: NW t mean -0.32 (sd 0.97), |t|>=2 in 1/8


planted (edge=0.6): NW t = +20.79, Welch t = +17.91


C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)


## Verdict

- **Signal — None.** Hou's industry-leader lead-lag does **not** replicate on 50 liquid US mega-caps: the long-up-leader / short-down-leader followers spread is **-3.64 bps/week** (NW *t* = **-0.77**) — indistinguishable from zero and on the *wrong* side if anything (Welch *t* = -1.99), unremarkable against a 1,000-permutation placebo (p = 0.93), sign-flipping across eras (-14.0 / +6.0 bps), and unchanged under a dollar-volume leader re-designation (-3.87 bps). The 20-seed synthetic control recovers a *planted* diffusion cleanly (*t* = +20.79, fires on 0/20 nulls), so this is a true absence — the effect lives among small illiquid firms this survivor panel omits.
- **Tradability — Mirage.** The specified book loses money gross and net (-6.60 bps/week at 1 bp, -14.60 at 5 bps); even the data-mined sign-flip is eaten by the 2.96 bps/week friction at 1 bp.